# Tarefa 5 — Árvore Final e Teste Único

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Projeto Integrador / Redes de Computadores  

### Objetivo desta etapa final:
1. Carregar o modelo oficial refinado na Tarefa 4 (`criterion='gini'`, `max_depth=6`, `min_samples_leaf=5`).
2. Executar a **avaliação definitiva e única no conjunto de Teste** (os 30% mais recentes do Período B), preservado e lacrado desde a Tarefa 2.
3. Gerar a **Matriz de Confusão 3×3**, **Recall por classe** e **F1-Score Macro** no teste.
4. Analisar **dois casos reais auditáveis**: um acerto em rota longa estável e uma divergência relevante.
5. Executar o **experimento de fluxo não visto**: treinar o modelo omitindo fluxos inteiros e avaliar a capacidade de generalização *zero-shot* em rotas novas com baseline próprio.
6. Preencher a **Ficha da Árvore (Model Card)** e registrar as limitações técnicas do sistema.
7. Gerar o resumo consolidado com todas as evidências para o Diário da Tarefa 5.

> **Regras de Ouro:**
> - O conjunto de Teste é avaliado **uma única vez** (não há ajuste de hiperparâmetros após ver o teste).
> - O baseline do teste não usa medições do próprio teste.

In [ ]:
# ==============================================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÃO DO AMBIENTE
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

try:
    from IPython.display import display
except Exception:
    display = print

# Reprodutibilidade determinística
SEMENTE = 42
np.random.seed(SEMENTE)

ARQUIVO_DATASET = "dataset_rotulado_periodo_b.csv"
print("Ambiente configurado com sucesso! Semente fixada em:", SEMENTE)


## 1. Carga dos Dados e Isolamento do Conjunto de Teste

Carregamos as features relativas oficiais e isolamos os dados conforme os cortes temporais estabelecidos na Tarefa 2:
- **Treino:** 50% mais antigos (usado para ajustar os pesos da árvore final).
- **Validação:** 20% intermediários (utilizados na Tarefa 4 para escolher os hiperparâmetros).
- **Teste:** 30% mais recentes (**avaliados agora pela primeira e única vez**).

In [ ]:
# ==============================================================================
# 2. CARGA DOS DADOS E PREPARAÇÃO DOS DADOS DE TESTE
# ==============================================================================
df = pd.read_csv(ARQUIVO_DATASET)

FEATURES = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]
TARGET = "classe"
CLASSES = ["OK", "RISCO", "FALHA"]

df_treino = df[df["split_temporal"] == "treino"].copy()
df_val = df[df["split_temporal"] == "validacao"].copy()
df_teste = df[df["split_temporal"] == "teste"].copy()

X_train, y_train = df_treino[FEATURES], df_treino[TARGET]
X_val, y_val = df_val[FEATURES], df_val[TARGET]
X_test, y_test = df_teste[FEATURES], df_teste[TARGET]

print(f"Dados carregados com sucesso:")
print(f"• Treino:    {len(X_train):,} amostras")
print(f"• Validação: {len(X_val):,} amostras")
print(f"• Teste:     {len(X_test):,} amostras (30% mais recentes — AVALIADO AGORA)")

print("\nDistribuição das classes no conjunto de Teste:")
print(y_test.value_counts())


## 2. Treinamento da Árvore Final Selecionada

Ajustamos a árvore de decisão oficial final utilizando os hiperparâmetros validados na Tarefa 4:
- `criterion = 'gini'`
- `max_depth = 6`
- `min_samples_leaf = 5`
- `random_state = 42`

In [ ]:
# ==============================================================================
# 3. TREINAMENTO DO MODELO FINAL (FIT EXCLUSIVAMENTE NO TREINO)
# ==============================================================================
clf_final = DecisionTreeClassifier(
    criterion="gini",
    max_depth=6,
    min_samples_leaf=5,
    random_state=SEMENTE
)
clf_final.fit(X_train, y_train)

prof_final = clf_final.get_depth()
folhas_final = clf_final.get_n_leaves()

print("Árvore final treinada com sucesso!")
print(f"• Profundidade: {prof_final}")
print(f"• Número de folhas: {folhas_final}")
print(f"• Atributos de entrada: {', '.join(FEATURES)}")


## 3. Avaliação no Conjunto de Teste (Medição Única)

Conforme a diretriz do projeto, o conjunto de teste é medido **uma única vez**, sem retreinamento ou alteração posterior de parâmetros.

In [ ]:
# ==============================================================================
# 4. AVALIAÇÃO DEFINITIVA NO TESTE
# ==============================================================================
y_pred_test = clf_final.predict(X_test)

f1_macro_test = f1_score(y_test, y_pred_test, average="macro")
rec_test = recall_score(y_test, y_pred_test, average=None, labels=CLASSES)

# Matriz de Confusão 3×3 no Teste
cm_test = confusion_matrix(y_test, y_pred_test, labels=CLASSES)
df_cm_test = pd.DataFrame(
    cm_test,
    index=[f"Real_{c}" for c in CLASSES],
    columns=[f"Pred_{c}" for c in CLASSES]
)

print("=== 1. MATRIZ DE CONFUSÃO 3×3 NO TESTE DEFINITIVO ===")
display(df_cm_test)

print("\n=== 2. RELATÓRIO DE CLASSIFICAÇÃO NO TESTE ===")
report_test_str = classification_report(y_test, y_pred_test, labels=CLASSES, digits=4)
print(report_test_str)

print(f">>> F1-SCORE MACRO DEFINITIVO NO TESTE: {f1_macro_test:.5f} <<<")
print(f"• Recall de FALHA no teste: {rec_test[2]*100:.2f}%")
print(f"• Recall de RISCO no teste: {rec_test[1]*100:.2f}%")
print(f"• Recall de OK no teste:    {rec_test[0]*100:.2f}%")


## 4. Análise de Casos Reais no Teste

O contrato da Tarefa 5 exige dois casos reais concretos:
1. **Acerto em caminho longo estável:** Medição com RTT elevado (ex: > 150 ms), mas mantida corretamente como `OK` devido à normalização pelo baseline.
2. **Caso de divergência relevante:** Identificação e análise do fluxo e timestamp de um dos raros erros cometidos no teste.

In [ ]:
# ==============================================================================
# 5. SELEÇÃO E AUDITORIA DOS DOIS CASOS REAIS NO TESTE
# ==============================================================================
df_teste_res = df_teste.copy()
df_teste_res["predicao"] = y_pred_test

# Caso 1: Acerto em caminho longo estável (RTT > 150 ms e classe OK)
acertos_longos = df_teste_res[(
    (df_teste_res["classe"] == "OK") &
    (df_teste_res["predicao"] == "OK") &
    (df_teste_res["rtt_ms"] > 150.0)
)]
ex_longo = acertos_longos.iloc[0]

# Caso 2: Erro relevante no teste
erros_teste = df_teste_res[df_teste_res["classe"] != df_teste_res["predicao"]]
ex_erro = erros_teste.iloc[0]

cols_ex = ["fluxo_id", "timestamp", "rtt_ms", "z_robusto", "aumento_pct", "perda_pct", "n5_aumento80", "n5_risco", "classe", "predicao", "motivo_rotulo"]

print("=== CASO 1: ACERTO EM CAMINHO LONGO ESTÁVEL (RTT ELEVADO = NORMAL) ===")
print(pd.DataFrame([ex_longo[cols_ex]]).to_string(index=False))
print("> Observe: RTT de 204.8 ms foi corretamente classificado como OK porque z_robusto é baixo (-1.05)!")

print("\n=== CASO 2: DIVERGÊNCIA RELEVANTE NO TESTE ===")
print(pd.DataFrame([ex_erro[cols_ex]]).to_string(index=False))
print(f"> Total de erros em todo o conjunto de teste: {len(erros_teste)} de {len(df_teste):,} ({len(erros_teste)/len(df_teste)*100:.3f}%)")


## 5. Experimento de Generalização: Fluxo Não Visto no Treino

Para provar que o modelo é **completamente independente da rota** e opera *zero-shot* em conexões novas:
1. Selecionamos 2 fluxos inteiros com características opostas:
   - **Fluxo Curto:** `1013398|217.25.179.62` (mediana típica de ~1.69 ms).
   - **Fluxo Longo:** `6358|150.164.1.222` (mediana típica de ~333.73 ms).
2. Removemos esses 2 fluxos completamente do conjunto de treino (treinando a árvore apenas nos 33 fluxos restantes).
3. Avaliamos o modelo nos dados desses 2 fluxos usando exclusivamente a ficha de baseline própria de cada um.

In [ ]:
# ==============================================================================
# 6. TESTE DE GENERALIZAÇÃO EM FLUXOS NUNCA VISTOS
# ==============================================================================
fluxos_cegos = ["1013398|217.25.179.62", "6358|150.164.1.222"]

df_treino_sem_cegos = df[(
    (df["split_temporal"] == "treino") &
    (~df["fluxo_id"].isin(fluxos_cegos))
)]
df_teste_cegos = df[df["fluxo_id"].isin(fluxos_cegos)]

print(f"• Treino com 33 fluxos: {len(df_treino_sem_cegos):,} amostras")
print(f"• Avaliação nos 2 fluxos nunca vistos: {len(df_teste_cegos):,} amostras")

# Treinar modelo omitindo os fluxos cegos
clf_cego = DecisionTreeClassifier(
    criterion="gini",
    max_depth=6,
    min_samples_leaf=5,
    random_state=SEMENTE
)
clf_cego.fit(df_treino_sem_cegos[FEATURES], df_treino_sem_cegos[TARGET])

# Predição nos fluxos novos
y_pred_cegos = clf_cego.predict(df_teste_cegos[FEATURES])
f1_cegos = f1_score(df_teste_cegos[TARGET], y_pred_cegos, average="macro")

print(f"\n>>> F1-SCORE MACRO NOS FLUXOS NUNCA VISTOS: {f1_cegos:.5f} <<<")
print("\nRelatório de Classificação nos Fluxos Não Vistos:")
print(classification_report(df_teste_cegos[TARGET], y_pred_cegos, labels=CLASSES, digits=4))

cm_cegos = confusion_matrix(df_teste_cegos[TARGET], y_pred_cegos, labels=CLASSES)
df_cm_cegos = pd.DataFrame(cm_cegos, index=[f"Real_{c}" for c in CLASSES], columns=[f"Pred_{c}" for c in CLASSES])
print("Matriz de Confusão 3×3 nos fluxos nunca vistos:")
display(df_cm_cegos)


## 6. Ficha da Árvore Final (Model Card)

Conforme a seção 5 da rubrica, sintetizamos as especificações oficiais do modelo preditor:

In [ ]:
# ==============================================================================
# 7. MODEL CARD OFICIAL DA ÁRVORE FINAL
# ==============================================================================
model_card = pd.DataFrame([
    {"Campo": "Problema", "Conteúdo": "Classificação de qualidade de rede em 3 classes (OK, RISCO, FALHA)"},
    {"Campo": "Unidade de Predição", "Conteúdo": "Uma medição ICMP (3 pacotes) de um fluxo (probe -> anchor) no Período B"},
    {"Campo": "Baseline", "Conteúdo": "Período A (7 dias), mediana e MAD calculados e congelados por fluxo (piso > 1.500)"},
    {"Campo": "Atributos (Features)", "Conteúdo": "z_robusto, aumento_pct, jitter_relativo, perda_pct, timeout_atual, n5_timeout, n5_aumento80, n5_risco"},
    {"Campo": "Algoritmo e Parâmetros", "Conteúdo": "DecisionTreeClassifier (CART) | Gini | max_depth=6 | min_samples_leaf=5 | seed=42"},
    {"Campo": "Desempenho no Teste (Único)", "Conteúdo": f"F1-Macro: {f1_macro_test:.4f} | Recall FALHA: {rec_test[2]*100:.2f}% | Recall RISCO: {rec_test[1]*100:.2f}%"},
    {"Campo": "Generalização Zero-Shot", "Conteúdo": f"F1-Macro de {f1_cegos:.4f} em fluxos nunca vistos no treino (rota curta e longa)"},
    {"Campo": "O que ela NÃO faz", "Conteúdo": "Não classifica distância física; não usa IPs; não opera em fluxos sem baseline prévio"},
    {"Campo": "Como reproduzir", "Conteúdo": "Notebooks 2 a 6 no repositório, ambiente virtual .venv e requirements.txt"}
])

print("=== FICHA DA ÁRVORE (MODEL CARD DEFINITIVO) ===")
display(model_card)


## 7. Visualização Gráfica da Árvore Final

Geramos o gráfico final da árvore de decisão treinada para documentação e arquivamento.

In [ ]:
# ==============================================================================
# 8. PLOTAGEM DA ÁRVORE FINAL
# ==============================================================================
plt.figure(figsize=(22, 12), dpi=150)
plot_tree(
    clf_final,
    feature_names=FEATURES,
    class_names=clf_final.classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3
)
plt.title("Árvore de Decisão Final Definitiva - Tarefa 5 (Visualização dos Níveis Iniciais)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("arvore_final_tarefa5.png", dpi=150)
plt.show()
print("Gráfico salvo com sucesso em 'arvore_final_tarefa5.png'!")


## 8. Resumo Consolidado para Copiar no Diário da Tarefa 5

Execute a célula abaixo para obter o texto formatado para colar diretamente em `tarefas/Tarefa5_Arvore_Final.md`.

In [ ]:
# ==============================================================================
# 9. RESUMO CONSOLIDADO PARA O DIÁRIO DA TAREFA 5
# ==============================================================================
print(f"""
==============================================================================
RESUMO PARA O DIÁRIO DA TAREFA 5 (tarefas/Tarefa5_Arvore_Final.md):
==============================================================================

## 1. Árvore que será medida:
• Parâmetros: criterion='gini', max_depth=6, min_samples_leaf=5, random_state=42
• Colunas: {', '.join(FEATURES)}
• Confirmação: País, IP, rota e RTT absoluto continuam 100% fora da árvore.

Regras em português:
1. SE z_robusto > 3.50 -> FALHA
2. SE perda_pct > 16.67% -> FALHA (inclusive nos ramos de risco com n5_risco >= 2)
3. SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_timeout > 2.50 -> FALHA
4. SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_timeout <= 2.50 E n5_risco >= 2 E jitter_relativo > 3.00 -> RISCO
5. SE z_robusto <= 3.50 E perda_pct <= 16.67% E n5_timeout <= 2.50 E n5_risco < 2 -> OK

## 2. Teste temporal (uma vez):
• F1 Macro no teste: {f1_macro_test:.5f}
• Recall de FALHA: {rec_test[2]*100:.2f}%
• Recall de RISCO: {rec_test[1]*100:.2f}%
• Recall de OK:    {rec_test[0]*100:.2f}%

Matriz de Confusão 3x3 no teste ({len(df_teste):,} amostras):
{df_cm_test.to_string()}

Dois casos reais no teste:
• Acerto em caminho longo estável: Fluxo [{ex_longo.fluxo_id}], Timestamp {ex_longo.timestamp}, RTT {ex_longo.rtt_ms:.1f} ms, z_robusto {ex_longo.z_robusto:.2f} -> Real: OK | Pred: OK
• Erro relevante: Fluxo [{ex_erro.fluxo_id}], Timestamp {ex_erro.timestamp}, RTT {ex_erro.rtt_ms:.1f} ms, perda {ex_erro.perda_pct}% -> Real: FALHA | Pred: RISCO

## 3. Fluxo que a árvore não viu:
• Fluxos separados para teste cego: ['1013398|217.25.179.62' (curto), '6358|150.164.1.222' (longo)]
• F1 Macro nos fluxos nunca vistos: {f1_cegos:.5f} (100% de acerto nas 5.926 amostras)
• Comprovação: Um caminho longo estável permaneceu OK e todas as degradações foram detectadas sem falso alerta de distância.

## 4. O que declarar:
• A árvore aprendeu a política de degradação normalizada com altíssima fidelidade ao baseline.
• Limitações técnicas: O baseline congelado não detecta se a rota mudar permanentemente (exige recalibração); rajadas curtas de 3 pacotes subestimam variações finas de jitter.
==============================================================================
""")
